In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from typing import List, Tuple


In [2]:
# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

def generate_hard_set(n_samples: int = 10000, difficulty_level: str = 'hard') -> pd.DataFrame:
    """Generate a challenging dataset for rent classification with adjustable difficulty"""
    
    # Define difficulty weights
    difficulty_weights = {
        'easy': [0.4, 0.2, 0.05, 0.05, 0.05, 0.05, 0.1, 0.05, 0.05],
        'medium': [0.15, 0.15, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
        'hard': [0.1, 0.25, 0.15, 0.15, 0.15, 0.05, 0.05, 0.05, 0.05],
        'extreme': [0.05, 0.35, 0.2, 0.2, 0.1, 0.05, 0.0, 0.0, 0.05]
    }
    
    # Get weights for the specified difficulty level
    weights = difficulty_weights.get(difficulty_level, difficulty_weights['hard'])
    
    # 1. SYNONYMS & IMPLIED RENT (No direct keywords)
    rent_synonyms = [
        # Formal/legal terms
        "Lease obligation payment",
        "Tenancy remittance",
        "Residential occupancy fee",
        "Domicile disbursement",
        "Habitation remittance",
        "Premises maintenance remittance",
        "Abode disbursement",
        "Dwelling remittance",
        "Residence disbursement",
        # Corporate/institutional terms
        "Recurring habitation expenditure",
        "Periodic domicile remittance",
        "Monthly premises fee",
        "Housing allocation payment",
        "Residential facility fee",
        "Accommodation disbursement",
        "Lodging remittance",
        # Vague but contextually clear
        "Where I live payment",
        "Place I stay remittance",
        "My apartment payment",
        "My house monthly",
        "My unit disbursement",
        # Property management terms
        "Property management remittance",
        "Real estate management fee",
        "Landlord services payment",
        "Building administration fee",
        "Complex management remittance",
        "Facility management disbursement",
        # Invoice/reference based
        "Invoice #APT2024-001",
        "Ref: RES-UNIT-5B",
        "Payment ref: HOUSING-",
        "Invoice: RESIDENCE",
        "Remittance: DWELLING",
        # Address-based (no explicit rent)
        "123 Main St monthly",
        "500 Oak Ave payment",
        "For 303 Parkview",
        "Unit 7B monthly",
        "Apartment 12C fee"
    ]
    
    # 2. AMBIGUOUS TRANSACTIONS (Could be rent OR something else)
    ambiguous_phrases = [
        # Without context, could be anything
        "Monthly payment to John",
        "Recurring transfer to Smith",
        "ACH to Johnson LLC",
        "Check #1234 for monthly",
        "Debit to Anderson",
        "Wire to Thompson",
        "Zelle to Davis",
        "Transfer to Miller",
        "Payment to Wilson",
        "Remittance to Moore",
        # Property-related but ambiguous
        "Property Services LLC",
        "Real Estate Solutions",
        "Housing Associates",
        "Residential Services",
        "Building Management",
        "Facility Solutions",
        "Complex Services",
        "Unit Management",
        "Premises Services",
        "Domicile Solutions",
        # Amount/date patterns only
        "Payment on 1st of month",
        "Monthly on 15th",
        "Recurring 1st week",
        "First of month payment",
        "Mid-month transfer",
        "End of month remittance",
        "Periodic 30th payment",
        "Bi-weekly housing",
        "Semi-monthly residence",
        "Quarterly dwelling"
    ]
    
    # 3. HOMONYMS & POLYSEMY (Same word, different meanings)
    homonym_contexts = [
        # "Unit" could be apartment OR measurement/business unit
        "Unit payment",
        "Unit fee",
        "Unit charge",
        "Unit disbursement",
        "Unit remittance",
        # "Complex" could be apartment complex OR complicated
        "Complex fee",
        "Complex payment",
        "Complex charge",
        # "Building" could be rent OR construction
        "Building payment",
        "Building fee",
        "Building disbursement",
        # "Property" could be rent OR insurance/tax
        "Property payment",
        "Property fee",
        "Property remittance",
        # "Lease" could be apartment OR car/equipment
        "Lease payment",
        "Lease fee",
        "Lease charge"
    ]
    
    # 4. NEGATION & EXCEPTIONS
    negations = [
        "NOT rent - utilities",
        "Non-rent housing expense",
        "Excluding rent - HOA",
        "Besides rent - insurance",
        "Other than rent - tax",
        "Additional to rent - fee",
        "Separate from rent - deposit",
        "Independent of rent - repair",
        "Distinct from rent - service",
        "Apart from rent - maintenance"
    ]
    
    # 5. CULTURAL/LINGUISTIC VARIATIONS
    cultural_terms = [
        # British English
        "Letting fee",
        "Tenancy payment",
        "Lodgings remittance",
        "Digs payment",
        # Australian
        "Rental payment",
        "Flat fee",
        "Unit letting",
        # Canadian
        "Condo fee",
        "Strata payment",
        # Informal/slang
        "Pad payment",
        "Crib fee",
        "Spot payment",
        "Place remittance"
    ]
    
    # 6. COMPOUND/COMPLEX DESCRIPTIONS
    compounds = [
        "Monthly housing and utilities bundle",
        "Residence fee including amenities",
        "Apartment payment + parking",
        "Rent with included services",
        "Dwelling disbursement plus fees",
        "Habitation remittance with utilities",
        "Unit payment including maintenance",
        "Premises fee with service charge",
        "Abode payment plus insurance",
        "Accommodation remittance including tax"
    ]
    
    # 7. MISSPELLINGS & TYPOS
    misspellings = [
        "Rnet payment",
        "Rant fee",
        "Rentt disbursement",
        "Rant remittance",
        "Rent (missed auto)",
        "Ren tpayment",
        "R3nt fee",
        "Rent-paymnt",
        "Rent-pay ment",
        "Rent(payment)",
        "Rent..payment",
        "Rent ;payment",
        "Rent/payment"
    ]
    
    # 8. FORMAT VARIATIONS
    formats = [
        "RENT - APT 5B",
        "RENT: UNIT 304",
        "RENT PAYMENT (MONTHLY)",
        "RENT - DO NOT DELETE",
        "RENT #2024-001",
        "RENT_INVOICE_001",
        "RENT-RECURRING",
        "RENT~MONTHLY",
        "RENT|AUTO",
        "RENT>>PAYMENT"
    ]
    
    # 9. CONTEXT-DEPENDENT AMOUNTS
    # Rent-like amounts but could be other things
    rent_amounts = [650, 750, 850, 950, 1100, 1250, 1350, 1450, 1550, 1650, 1750, 1850, 1950, 2100, 2250, 2400, 2600, 2800, 3000]
    non_rent_amounts = [50, 75, 100, 125, 150, 200, 250, 300, 350, 400, 450, 500, 600, 700, 800, 900, 1000, 1200, 1400, 1600, 1800]
    
    # 10. TEMPORAL PATTERNS
    date_patterns = [
        "1st of month", "5th monthly", "10th recurring", "15th auto", "20th periodic",
        "25th standing", "Last day", "First Monday", "Bi-weekly", "Semi-monthly",
        "Quarterly", "Every 30 days", "Monthly anniversary", "Calendar month"
    ]
    
    # EXTREME MODE: Add even more ambiguous phrases
    if difficulty_level == 'extreme':
        # Add identical descriptions that could be either rent or not
        extreme_ambiguous = [
            "Recurring monthly payment",
            "Standard monthly charge",
            "Regular payment",
            "Monthly obligation",
            "Periodic disbursement",
            "Scheduled transfer",
            "Automated monthly payment",
            "Standing order payment",
            "Recurring debit",
            "Monthly auto-payment"
        ]
        # Extend ambiguous phrases for extreme mode
        ambiguous_phrases.extend(extreme_ambiguous)
        
        # Make amounts more overlapping
        rent_amounts = list(range(600, 3100, 50))  # More overlap
        non_rent_amounts = list(range(100, 3000, 50))  # Same range for confusion
    
    # Generate the dataset
    data = []
    # Track used descriptions to avoid duplicates
    used_descriptions = set()
    
    for i in range(n_samples):
        # For balanced dataset
        is_rent = random.choice([True, False])
        
        # Generate date (spanning 2 years)
        start_date = datetime(2024, 1, 1)
        days_diff = random.randint(0, 730)
        current_date = start_date + timedelta(days=days_diff)
        
        # Select description category based on difficulty level
        difficulty = random.choices(
            ['synonym', 'ambiguous', 'homonym', 'negation', 'cultural', 
             'compound', 'misspelling', 'format', 'temporal'],
            weights=weights
        )[0]
        
        description = ""
        
        if difficulty == 'synonym':
            base_desc = random.choice(rent_synonyms) if is_rent else random.choice(ambiguous_phrases)
        elif difficulty == 'ambiguous':
            # EXTREME CHALLENGE: For extreme difficulty, use same descriptions for both classes
            if difficulty_level == 'extreme' and random.random() < 0.4:
                # 40% of ambiguous cases use identical text for both rent/non-rent
                base_desc = random.choice(ambiguous_phrases)
                # Decision based on OTHER factors (date, amount patterns)
                if current_date.day == 1 and random.random() < 0.7:
                    is_rent = True  # 1st of month payments are often rent
                elif 1400 <= random.choice(rent_amounts) <= 2200 and random.random() < 0.6:
                    is_rent = True  # Typical rent range
                else:
                    is_rent = random.choice([True, False])  # Otherwise random
            else:
                base_desc = random.choice(ambiguous_phrases)
                
            # Add subtle contextual clues for rent
            if is_rent and random.random() < 0.3:
                base_desc += f" (Unit {random.randint(1, 50)}{random.choice(['A', 'B', 'C'])})"
        elif difficulty == 'homonym':
            base_desc = random.choice(homonym_contexts)
            # Rent clues in amount/date
            if is_rent:
                base_desc += f" for residential unit"
        elif difficulty == 'negation':
            if is_rent:
                base_desc = f"Rent payment {random.choice(['', '(primary residence)', '(apartment)'])}"
            else:
                base_desc = random.choice(negations)
        elif difficulty == 'cultural':
            base_desc = random.choice(cultural_terms)
        elif difficulty == 'compound':
            base_desc = random.choice(compounds) if is_rent else f"{random.choice(['Utility', 'Service', 'Maintenance', 'HOA'])} fee {random.choice(['+ tax', '+ insurance', 'incl. services'])}"
        elif difficulty == 'misspelling':
            if is_rent:
                base_desc = random.choice(misspellings)
            else:
                base_desc = f"{random.choice(['Utilitie', 'Servise', 'Maintenence', 'Subscripton'])} payment"
        elif difficulty == 'format':
            base_desc = random.choice(formats) if is_rent else f"{random.choice(['UTILITY', 'SERVICE', 'FEE', 'CHARGE'])} - {random.choice(['MONTHLY', 'RECURRING', 'AUTO'])}"
        elif difficulty == 'temporal':
            temporal = random.choice(date_patterns)
            if is_rent:
                base_desc = f"Residential payment {temporal}"
            else:
                base_desc = f"{random.choice(['Service', 'Subscription', 'Membership'])} {temporal}"
        else:
            base_desc = "Transaction payment"
        
        # Add amount noise - EXTRA HARD for extreme mode
        if is_rent:
            base_amount = random.choice(rent_amounts)
            # Add small variations
            amount = base_amount + random.choice([-50, -25, 0, 25, 50])
        else:
            amount = random.choice(non_rent_amounts)
            # Sometimes use rent-like amounts for non-rent (challenge!)
            if difficulty_level in ['hard', 'extreme']:
                # More often use rent-like amounts in harder modes
                if random.random() < 0.4:  # 40% chance in hard/extreme
                    amount = random.choice(rent_amounts)
            elif random.random() < 0.2:  # 20% chance in easier modes
                amount = random.choice(rent_amounts)
        
        # Format amount (negative for payments)
        formatted_amount = f"-${amount:,.2f}" if random.random() < 0.9 else f"${amount:,.2f}"
        
        # Generate realistic balance
        balance = random.randint(1000, 50000)
        
        # Generate a unique transaction ID and ensure description uniqueness
        while True:
            # Generate unique identifier components
            trans_id = random.randint(100000, 999999)
            date_str = current_date.strftime('%Y%m%d')
            suffix = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', k=4))
            
            # Append unique identifier to base description
            # Choose different formats for variety
            identifier_format = random.choice([
                f" [Ref:{trans_id}]",
                f" [ID:{trans_id}]",
                f" [TXN:{trans_id}]",
                f" #{trans_id}",
                f" [Date:{date_str}]",
                f" [Ref:{suffix}]",
                f" [ID:{date_str}-{trans_id % 1000}]",
                f" [Trans:{trans_id}]",
                f" [Invoice:{trans_id}]",
                f" [Payment:{suffix}]"
            ])
            
            # Sometimes add timestamp for more uniqueness
            if random.random() < 0.2:
                hour = random.randint(0, 23)
                minute = random.randint(0, 59)
                identifier_format += f" [{hour:02d}:{minute:02d}]"
            
            unique_description = base_desc + identifier_format
            
            # Check if this description is already used
            if unique_description not in used_descriptions:
                used_descriptions.add(unique_description)
                break
        
        # Create transaction record
        record = {
            'Date': current_date.strftime('%m/%d/%Y'),
            'Description': unique_description,
            'Comments': '',  # Empty for simplicity
            'Check Number': '',  # Empty for simplicity
            'Amount': formatted_amount,
            'Balance': f"${balance:,.2f}",
            'rent': 1 if is_rent else 0
        }
        
        data.append(record)
    
    return pd.DataFrame(data)

# Example usage with different difficulties:
print("Generating datasets with different difficulty levels...")

# Generate datasets
df_easy = generate_hard_set(n_samples=1000, difficulty_level='easy')
df_medium = generate_hard_set(n_samples=1000, difficulty_level='medium')
df_hard = generate_hard_set(n_samples=1000, difficulty_level='hard')
df_extreme = generate_hard_set(n_samples=1000, difficulty_level='extreme')

# Save them
df_easy.to_csv('rent_dataset_easy_1000.csv', index=False)
df_medium.to_csv('rent_dataset_medium_1000.csv', index=False)
df_hard.to_csv('rent_dataset_hard_1000.csv', index=False)
df_extreme.to_csv('rent_dataset_extreme_1000.csv', index=False)

print("Datasets saved:")
print("  - rent_dataset_easy_1000.csv")
print("  - rent_dataset_medium_1000.csv")
print("  - rent_dataset_hard_1000.csv")
print("  - rent_dataset_extreme_1000.csv")

# Show sample from each difficulty
def show_sample(df, difficulty):
    rent_count = df['rent'].sum()
    keyword_count = sum('rent' in desc.lower() for desc in df['Description'])
    
    print(f"\n{difficulty.upper()} DATASET (n={len(df)})")
    print(f"  Rent transactions: {rent_count} ({rent_count/len(df)*100:.1f}%)")
    print(f"  Contains 'rent' keyword: {keyword_count} ({keyword_count/len(df)*100:.1f}%)")
    print(f"  Average description length: {df['Description'].str.len().mean():.1f} chars")
    
    # Check for duplicates
    unique_descriptions = df['Description'].nunique()
    print(f"  Unique descriptions: {unique_descriptions} ({(unique_descriptions/len(df))*100:.1f}% unique)")
    
    # Show 5 sample transactions
    print("\n  Sample transactions:")
    for idx, row in df.sample(5, random_state=42).iterrows():
        label = "RENT" if row['rent'] == 1 else "NOT RENT"
        print(f"    {row['Description'][:60]:60} | {row['Amount']:>10} | {label}")

show_sample(df_easy, 'easy')
show_sample(df_medium, 'medium')
show_sample(df_hard, 'hard')
show_sample(df_extreme, 'extreme')

# Verify no duplicates exist
print("\n\n=== VERIFYING NO DUPLICATE DESCRIPTIONS ===")
for name, df in [('easy', df_easy), ('medium', df_medium), ('hard', df_hard), ('extreme', df_extreme)]:
    duplicates = df['Description'].duplicated().sum()
    print(f"{name.capitalize()} dataset: {duplicates} duplicate descriptions found")
    
    # If duplicates found, show them
    if duplicates > 0:
        dup_descriptions = df[df['Description'].duplicated(keep=False)]['Description'].unique()
        print(f"  Example duplicate: {dup_descriptions[0][:50]}..." if len(dup_descriptions) > 0 else "")

Generating datasets with different difficulty levels...
Datasets saved:
  - rent_dataset_easy_1000.csv
  - rent_dataset_medium_1000.csv
  - rent_dataset_hard_1000.csv
  - rent_dataset_extreme_1000.csv

EASY DATASET (n=1000)
  Rent transactions: 489 (48.9%)
  Contains 'rent' keyword: 105 (10.5%)
  Average description length: 35.6 chars
  Unique descriptions: 1000 (100.0% unique)

  Sample transactions:
    Residential Services [TXN:680990]                            | -$1,800.00 | NOT RENT
    Unit Management [TXN:372735]                                 |   -$800.00 | NOT RENT
    Payment to Wilson [Ref:760957] [16:55]                       |   -$350.00 | NOT RENT
    Residential payment 1st of month [Trans:404779]              | -$1,500.00 | RENT
    Rent with included services [Ref:818530] [11:18]             | -$1,500.00 | RENT

MEDIUM DATASET (n=1000)
  Rent transactions: 493 (49.3%)
  Contains 'rent' keyword: 198 (19.8%)
  Average description length: 36.0 chars
  Unique description